# 02 Can neighbours predict the missing hours?
Hide the hours of an area we do know, copy them from the nearest known area, and measure.

`sjoin_nearest` is index-backed, so this runs over every area rather than a sample.

In [ ]:
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt

areas = gpd.read_parquet("../data/processed/parking_rules.parquet")  # 8,754 areas, metres
print(len(areas), "areas | CRS", areas.crs.to_epsg())

In [ ]:
known = areas[areas["hours"].notna()]
missing = areas[areas["status"] == "missing_hours"]
print(len(known), "with hours |", len(missing), "missing")

## Accuracy of copying the nearest neighbour

In [ ]:
def nearest_match(pool, same_class):
    """Pair every area with its nearest other area and check whether the hours agree."""
    groups = pool.groupby("luokka") if same_class else [(None, pool)]
    out = [gpd.sjoin_nearest(g[["geometry", "hours"]], g[["geometry", "hours"]],
                             exclusive=True, distance_col="dist")
           for _, g in groups if len(g) > 1]
    paired = pd.concat(out)
    paired["match"] = paired["hours_left"] == paired["hours_right"]
    return paired

res = {s: nearest_match(known, s) for s in (False, True)}
pd.Series({f"same_class={s}": r["match"].mean() for s, r in res.items()})

Accuracy falls with distance, which is what the confidence threshold will be built on.

In [ ]:
for same, r in res.items():
    bins = pd.cut(r["dist"], [0, 10, 50, 200, 1e9])
    print("same_class =", same)
    print(r.groupby(bins, observed=True)["match"].agg(["mean", "size"]).to_string(), "\n")

## The catch
Areas with missing hours sit far from areas with known hours, so the figures above are optimistic.

In [ ]:
gap = pd.concat([gpd.sjoin_nearest(m[["geometry"]], known[known["luokka"] == k][["geometry"]],
                                    distance_col="dist")
                 for k, m in missing.groupby("luokka") if (known["luokka"] == k).any()])
print(gap["dist"].describe())
print("within 50 m:", f"{(gap['dist'] <= 50).mean():.1%}")